In [ ]:
from openai import OpenAI
from anthropic import Anthropic
from dotenv import load_dotenv
import os
import csv
import pandas as pd
import json

# Load the environment variables
load_dotenv()

# Initialize the OpenAI client
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [ ]:
def parse_job_postings(job_content: str, job_title: str, company: str):
    prompt = f"""Parse this job posting and extract metadata in JSON format.

    Job Title: {job_title}
    Company: {company}
    Job Description: {job_content}

    Extract the following information and return ONLY valid JSON (no markdown, no backticks, no explanation):

    {{
        "location_type": "remote" | "hybrid" | "onsite" | unknown,
        "location": "city, state/country" or null,
        "salary_min": number or null,
        "salary_max": number or null,
        "industry": "Tech" | "Finance" | "Marketing" | "Sales" | "Engineering" | "Design" | "Other" etc or null
    }}

    Rules:
    1. For location_type, look for keywords: "remote", "hybrid", "onsite", "in-office", "work from home"
    2. Extract salary even if it's a range. Convert to numbers (no $ or commas)
    3. If information is not clearly stated, use null.
    4. DO NOT include any text outside the JSON object
    5. DO NOT use markdown code blocks or backticks
    """

    try:
        message = anthropic_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=2000,
            messages=[{"role": "user", "content": prompt}]
        )

        response_text = message.content[0].text.strip()
        metadata = json.loads(response_text)

        print(metadata)
        return metadata

    except json.JSONDecodeError as e:
        print(f"JSON parsing error: {e}")
        return None

    except Exception as e:
        print(f"Error parsing job: {e}")
        return None

        


In [ ]:
with open("job_postings.csv", "r") as file:
    reader = csv.reader(file)
    next(reader)  # Skip the header row
    for row in reader:
        company = row[0]
        url = row[1]
        job_title = row[2]  # title is at index 2
        job_content = row[3]  # content is at index 3
        metadata = parse_job_postings(job_content, job_title, company)
        print(url)
        break
